# Falcon Perception Agent — Grounded Visual Reasoning

This cookbook demonstrates how to use **Falcon Perception** as a grounded perception tool inside a multi-step visual reasoning agent.

## Architecture

```
User query + Image
        │
        ▼
 ┌─────────────────────────────────────────────────┐
 │      Orchestrator VLM  (GPT-5 mini / any VLM)   │
 │  Reasons about the query, plans tool calls,     │
 │  interprets structured feedback, synthesises    │
 │  a grounded final answer.                       │
 └──────────────┬──────────────────────────────────┘
                │  tool calls
      ┌─────────┴──────────────────────┐
      │                                │
      ▼                                ▼
 ground_expression            compute_relations
 ┌──────────────────┐          ┌─────────────────────┐
 │  Falcon          │          │  pycocotools + numpy │
 │  Perception      │          │  IoU, centroid,      │
 │  (segmentation)  │          │  size ratio, dist.   │
 └──────────────────┘          └─────────────────────┘
      │                                │
      │  SoM image + JSON metadata     │  JSON relations
      └─────────────────────┬──────────┘
                            ▼
                    Orchestrator VLM
                    (next step decision)
```

### What makes this different from a standard segmentation demo

- The orchestrator VLM **reasons** over the scene and decides what to segment.
- Falcon Perception handles **full referring expressions** (not just noun phrases).
- The agent gets both **visual feedback** (Set-of-Marks image) and **structured numerical feedback** (area, centroid, bbox, IoU).
- The loop continues until the VLM is confident, using as many FP calls as needed.

### Available tools

| Tool | When to use | Returns |
|---|---|---|
| `ground_expression(expr)` | Segment everything matching a referring expression | SoM image + JSON metadata |
| `get_crop(mask_id)` | Zoom into a small or overlapping mask | Cropped image |
| `compute_relations(mask_ids)` | Get precise spatial facts between masks | JSON (IoU, position, size) |
| `answer(response, mask_ids)` | Return the final answer and terminate | — |

## 1 — Install dependencies

In [ ]:
# Run once if needed
# !uv pip install openai pillow numpy pycocotools scipy

## 2 — Imports and path setup

In [ ]:
import sys
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image

# Make both Falcon-Perception root and demo/ importable
_DEMO_DIR = Path(".").resolve()           # demo/
_ROOT_DIR = _DEMO_DIR.parent              # Falcon-Perception/
for p in [str(_ROOT_DIR), str(_DEMO_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from falcon_perception import (
    PERCEPTION_MODEL_ID,
    load_and_prepare_model,
    setup_torch_config,
)
from falcon_perception.data import ImageProcessor
from falcon_perception.paged_inference import PagedInferenceEngine, engine_config_for_gpu

from agent import AgentResult, VLMClient, run_agent, run_baseline

setup_torch_config()
print("Imports OK.")

In [ ]:
# ── Model source: pick one ──────────────────────────────────────────────────
HF_MODEL_ID   = PERCEPTION_MODEL_ID   # "tiiuae/Falcon-Perception" (auto-downloaded)
HF_LOCAL_DIR  = None                  # or set to a local export path, e.g. "./my_export/"

# ── Inference settings ──────────────────────────────────────────────────────
DEVICE    = "cuda"
DTYPE     = "bfloat16"      # bfloat16 recommended for GPU inference
COMPILE   = True            # torch.compile — first call is slower, subsequent calls fast
CUDAGRAPH = True            # CUDA graphs for decode
MAX_DIM   = 1024            # longest-edge cap fed to FP

model, tokenizer, model_args = load_and_prepare_model(
    hf_model_id=HF_MODEL_ID,
    hf_local_dir=HF_LOCAL_DIR,
    device=DEVICE,
    dtype=DTYPE,
    compile=COMPILE,
)

image_processor = ImageProcessor(patch_size=16, merge_size=1)
cfg = engine_config_for_gpu(max_image_size=MAX_DIM, dtype=model.dtype)
print("Engine config:", cfg)

fp_engine = PagedInferenceEngine(
    model,
    tokenizer,
    image_processor,
    max_seq_length=model_args.max_seq_len,
    capture_cudagraph=CUDAGRAPH,
    **cfg,
)
print("Falcon Perception engine ready.")

## 4 — Configure the orchestrator VLM (GPT-5 mini)

Provide your OpenAI API key below. Any GPT-5 mini or GPT-5.4 mini model works.

To use a **self-hosted model** (e.g. Qwen-VL via vLLM), set `VLM_BASE_URL` to
your server URL, e.g. `"http://localhost:8000/v1"`.

In [ ]:
OPENAI_API_KEY = ""            # ← replace with your key
VLM_MODEL      = "gpt-5-mini"        # or "gpt-5-mini-2025-08-07", "gpt-5.4-mini", etc.
VLM_BASE_URL   = None                # None = default OpenAI endpoint

vlm_client = VLMClient(
    api_key=OPENAI_API_KEY,
    model=VLM_MODEL,
    base_url=VLM_BASE_URL,
    max_tokens=32000,
)
print(f"VLM client ready (model={VLM_MODEL}).")

## 5 — Warmup

The first FP inference call absorbs the `torch.compile` cost. Run this once
before the examples so that subsequent agent calls are fast.

In [ ]:
from agent.fp_tools import run_ground_expression

print("Running warmup inference (absorbs torch.compile cost) ...")
_warmup_img = Image.new("RGB", (512, 512), color=(128, 128, 128))
with torch.inference_mode():
    run_ground_expression(fp_engine, tokenizer, _warmup_img, "object",
                          max_dimension=MAX_DIM)
print("Warmup done. Agent is ready.")

---

## Examples

Each example runs the **Falcon Perception Agent** and a **plain VLM baseline**
(same model, no tools, no grounding) side-by-side so you can see exactly where
the structured perception feedback makes a difference.

### Helper utilities

In [ ]:
import io
import urllib.request


def load_image(source: str) -> Image.Image:
    """Load a PIL image from a local file path or HTTP(S) URL."""
    if source.startswith("http"):
        with urllib.request.urlopen(source) as resp:
            return Image.open(io.BytesIO(resp.read())).convert("RGB")
    return Image.open(source).convert("RGB")


def show_result(result: AgentResult, title: str = "") -> None:
    """Print agent answer + supporting mask image."""
    if title:
        print(f"\n{'=' * 60}")
        print(f"  {title}")
        print(f"{'=' * 60}")
    print(f"  Answer       : {result.answer}")
    print(f"  Support masks: {result.supporting_mask_ids}")
    print(f"  FP calls     : {result.n_fp_calls}")
    print(f"  VLM calls    : {result.n_vlm_calls}")
    if result.final_image is not None:
        display(result.final_image)


def show_comparison(result: AgentResult, baseline_answer: str, query: str) -> None:
    """Print a side-by-side comparison of agent vs plain-VLM baseline."""
    sep = "-" * 58
    print(f"\n{'=' * 60}")
    print(f"  Query: {query}")
    print(f"{'=' * 60}")
    print(f"\n  BASELINE  (VLM only, no Falcon Perception)")
    print(f"  {sep}")
    for line in (baseline_answer or "(no response)").splitlines():
        print(f"  {line}")
    print(f"\n  AGENT  (VLM + Falcon Perception grounding)")
    print(f"  {sep}")
    for line in result.answer.splitlines():
        print(f"  {line}")
    print(f"  Support masks : {result.supporting_mask_ids}")
    print(f"  FP calls      : {result.n_fp_calls}  |  VLM calls: {result.n_vlm_calls}")
    if result.final_image is not None:
        print()
        display(result.final_image)

---

### Example 1 — Spatial reasoning: *Which duck is flying the highest?*

The agent grounds all ducks, reads `centroid_norm.y` from the metadata
(smaller `y` = higher in frame), and picks the correct one.
The baseline VLM has to judge height purely from the raw image.

In [ ]:
image_1 = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/ducks.jpg")

In [ ]:
QUERY_1 = "Which duck is flying the highest?"

with torch.inference_mode():
    result_1 = run_agent(
        image_1, QUERY_1, fp_engine, tokenizer, vlm_client, verbose=True,
    )

baseline_1 = run_baseline(image_1, QUERY_1, vlm_client)
show_comparison(result_1, baseline_1, QUERY_1)

---

### Example 2 — Proximity ranking: *Which bird is closest to the camera?*

Proximity is estimated from `area_fraction` and vertical centroid position.
The agent grounds all birds and uses `compute_relations` to rank them
numerically; the baseline has to judge proximity visually.

In [ ]:
image_2 = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/seagull.jpg")

In [ ]:
QUERY_2 = "Which bird is closest to the camera?"

with torch.inference_mode():
    result_2 = run_agent(
        image_2, QUERY_2, fp_engine, tokenizer, vlm_client, verbose=True,
    )

baseline_2 = run_baseline(image_2, QUERY_2, vlm_client)
show_comparison(result_2, baseline_2, QUERY_2)

In [ ]:
MY_IMAGE = "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/image-135.jpg"   # local path or URL
MY_QUERY = "How many HP boxes are there? double check the results from Falcon Perception, DO NOT TRUST ENTIRELY"

my_image = load_image(MY_IMAGE)
print("Running agent mode ...")
with torch.inference_mode():
    my_result = run_agent(
        my_image, MY_QUERY, fp_engine, tokenizer, vlm_client, verbose=True,
    )
print("Running baseline ...")
my_baseline = run_baseline(my_image, MY_QUERY, vlm_client)
show_comparison(my_result, my_baseline, MY_QUERY)